# Module 4: Build a Word Game Environment

Build a letter-guessing (Hangman-style) environment from scratch using the OpenEnv pattern.

**Time:** ~30 min · **Difficulty:** Intermediate · **GPU:** Not required

In [ ]:
# Install dependencies
!uv pip install -q openenv-core==0.2.2

In [ ]:
import os

# Clean up any previous word_game directory to ensure a fresh start
!rm -rf word_game

# Initialize a new OpenEnv project structure using 'python -m openenv' to ensure it's found
!python -m openenv init word_game

print("OpenEnv project initialized in 'word_game' directory.")

/usr/bin/python3: No module named openenv
OpenEnv project initialized in 'word_game' directory.


### Create and Update Environment Files

Now we will write the necessary Python files for our OpenEnv application within the `word_game` directory. This includes `models.py`, `server/environment.py`, and `server/app.py`.

### Local Word Game Environment (Simplified)

In [ ]:
import os

# Content for word_game/models.py
models_content = '''
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

@dataclass
class WordGameAction:
    guess: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameObservation:
    done: bool
    reward: Optional[float]
    masked_word: str
    guessed_letters: List[str]
    attempts_remaining: int
    message: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameState:
    episode_id: Optional[str] = None
    step_count: int = 0
    target_word: str = ""
    max_attempts: int = 6
'''

# Content for word_game/server/environment.py
environment_content = '''
import random
import uuid
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

# Import models from the main models.py (which is handled by openenv during build)
# For local execution and type hinting within this file, we redefine them or import from a local stub

# --- Word Game Type Definitions ---
@dataclass
class WordGameAction:
    """Player guesses a single letter."""
    guess: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameObservation:
    """What the player sees after each guess."""
    done: bool
    reward: Optional[float]
    masked_word: str            # e.g., "p_th_n"
    guessed_letters: List[str]  # All letters tried
    attempts_remaining: int
    message: str                # Feedback text
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameState:
    """Episode metadata."""
    episode_id: Optional[str] = None
    step_count: int = 0
    target_word: str = ""
    max_attempts: int = 6

# --- Word Game Environment ---
WORDS = [
    "python", "neural", "tensor", "matrix", "vector",
    "kernel", "lambda", "signal", "binary", "cipher",
    "model", "layer", "epoch", "batch", "token",
]

class WordGameEnvironment:
    """A letter-guessing game environment following the OpenEnv pattern."""

    def __init__(self):
        self._state = WordGameState()
        self._target = ""
        self._guessed = set()
        self._remaining = 6

    def reset(self) -> WordGameObservation:
        """Start a new episode with a random word."""
        self._target = random.choice(WORDS)
        self._guessed = set()
        self._remaining = 6
        self._state = WordGameState(
            episode_id=str(uuid.uuid4()),
            step_count=0,
            target_word=self._target,
            max_attempts=6,
        )
        return WordGameObservation(
            done=False,
            reward=None,
            masked_word=self._mask(),
            guessed_letters=[],
            attempts_remaining=self._remaining,
            message=f"Guess letters in a {len(self._target)}-letter word!",
        )

    def step(self, action: WordGameAction) -> WordGameObservation:
        """Process a letter guess."""
        letter = action.guess.lower().strip()
        self._state.step_count += 1

        # Already guessed?
        if letter in self._guessed:
            return WordGameObservation(
                done=False,
                reward=0.0,
                masked_word=self._mask(),
                guessed_letters=sorted(self._guessed),
                attempts_remaining=self._remaining,
                message=f"Already guessed '{letter}'. Try another.",
            )

        self._guessed.add(letter)

        if letter in self._target:
            message = f"'{letter}' is in the word!"
        else:
            self._remaining -= 1
            message = f"'{letter}' is not in the word."

        # Check win/lose
        masked = self._mask()
        won = "_" not in masked
        lost = self._remaining <= 0
        done = won or lost

        if won:
            reward = 1.0
            message = f"You got it! The word was '{self._target}'."
        elif lost:
            reward = 0.0
            message = f"Out of attempts. The word was '{self._target}'."
        else:
            reward = 0.0

        return WordGameObservation(
            done=done,
            reward=reward,
            masked_word=masked,
            guessed_letters=sorted(self._guessed),
            attempts_remaining=self._remaining,
            message=message,
        )

    @property
    def state(self) -> WordGameState:
        return self._state

    def _mask(self) -> str:
        """Show guessed letters, hide the rest."""
        return "".join(c if c in self._guessed else "_" for c in self._target)
'''

# Content for word_game/server/app.py
app_content = '''
from openenv.core.env_server import create_fastapi_app
from word_game.server.environment import WordGameEnvironment
from word_game.models import WordGameAction, WordGameObservation

app = create_fastapi_app(WordGameEnvironment, WordGameAction, WordGameObservation)
'''

# Write models.py
with open("word_game/models.py", "w") as f:
    f.write(models_content)
print("Created/Updated word_game/models.py")

# Create server directory if it doesn't exist
os.makedirs("word_game/server", exist_ok=True)

# Write server/environment.py
with open("word_game/server/environment.py", "w") as f:
    f.write(environment_content)
print("Created/Updated word_game/server/environment.py")

# Write server/app.py
with open("word_game/server/app.py", "w") as f:
    f.write(app_content)
print("Created/Updated word_game/server/app.py")

### Deploy to Hugging Face Spaces

Now that all the necessary files are in place and your `HF_TOKEN` is set, we can deploy the environment to a Hugging Face Space using `openenv push`.

**Important:** You need to replace `your-username/word-game` with your actual Hugging Face username and the desired repository name. The repository will be created if it doesn't exist.

In [ ]:
# Change into the word_game directory before running openenv push
# Note: This command will start the deployment process. It may take some time.
!cd word_game && openenv push --repo-id your-username/word-game

print("Deployment command executed. Check your Hugging Face profile for the status of the Space.")

### Creating `inference.py`

To meet the requirement of having a standalone `inference.py` script, I will combine the `WordGame` environment definition, the policy classes, and the evaluation function into a single Python file. This file will be written to disk, and then executed.

In [ ]:
%%writefile inference.py
import random
import uuid
import string
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

# --- Word Game Type Definitions (from 5PJ2MVnlV7xP) ---
@dataclass
class WordGameAction:
    """Player guesses a single letter."""
    guess: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameObservation:
    """What the player sees after each guess."""
    done: bool
    reward: Optional[float]
    masked_word: str            # e.g., "p_th_n"
    guessed_letters: List[str]  # All letters tried
    attempts_remaining: int
    message: str                # Feedback text
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameState:
    """Episode metadata."""
    episode_id: Optional[str] = None
    step_count: int = 0
    target_word: str = ""
    max_attempts: int = 6

# --- Word Game Environment (from 1Zrghkx_V7xR) ---
WORDS = [
    "python", "neural", "tensor", "matrix", "vector",
    "kernel", "lambda", "signal", "binary", "cipher",
    "model", "layer", "epoch", "batch", "token",
]

class WordGameEnvironment:
    """A letter-guessing game environment following the OpenEnv pattern."""

    def __init__(self):
        self._state = WordGameState()
        self._target = ""
        self._guessed = set()
        self._remaining = 6

    def reset(self) -> WordGameObservation:
        """Start a new episode with a random word."""
        self._target = random.choice(WORDS)
        self._guessed = set()
        self._remaining = 6
        self._state = WordGameState(
            episode_id=str(uuid.uuid4()),
            step_count=0,
            target_word=self._target,
            max_attempts=6,
        )
        return WordGameObservation(
            done=False,
            reward=None,
            masked_word=self._mask(),
            guessed_letters=[],
            attempts_remaining=self._remaining,
            message=f"Guess letters in a {len(self._target)}-letter word!",
        )

    def step(self, action: WordGameAction) -> WordGameObservation:
        """Process a letter guess."""
        letter = action.guess.lower().strip()
        self._state.step_count += 1

        # Already guessed?
        if letter in self._guessed:
            return WordGameObservation(
                done=False,
                reward=0.0,
                masked_word=self._mask(),
                guessed_letters=sorted(self._guessed),
                attempts_remaining=self._remaining,
                message=f"Already guessed '{letter}'. Try another.",
            )

        self._guessed.add(letter)

        if letter in self._target:
            message = f"'{letter}' is in the word!"
        else:
            self._remaining -= 1
            message = f"'{letter}' is not in the word."

        # Check win/lose
        masked = self._mask()
        won = "_" not in masked
        lost = self._remaining <= 0
        done = won or lost

        if won:
            reward = 1.0
            message = f"You got it! The word was '{self._target}'."
        elif lost:
            reward = 0.0
            message = f"Out of attempts. The word was '{self._target}'."
        else:
            reward = 0.0

        return WordGameObservation(
            done=done,
            reward=reward,
            masked_word=masked,
            guessed_letters=sorted(self._guessed),
            attempts_remaining=self._remaining,
            message=message,
        )

    @property
    def state(self) -> WordGameState:
        return self._state

    def _mask(self) -> str:
        """Show guessed letters, hide the rest."""
        return "".join(c if c in self._guessed else "_" for c in self._target)

# --- Policy Definitions and Evaluation Function (from 52aizQEBV7xU) ---
class RandomLetterPolicy:
    """Guess random unused letters."""
    name = "Random"

    def select_action(self, obs: WordGameObservation) -> WordGameAction:
        available = [c for c in string.ascii_lowercase if c not in obs.guessed_letters]
        return WordGameAction(guess=random.choice(available))


class FrequencyPolicy:
    """Guess by English letter frequency."""
    name = "Frequency"
    FREQ_ORDER = "etaoinshrdlcumwfgypbvkjxqz"

    def select_action(self, obs: WordGameObservation) -> WordGameAction:
        for letter in self.FREQ_ORDER:
            if letter not in obs.guessed_letters:
                return WordGameAction(guess=letter)
        return WordGameAction(guess="a")  # fallback


def evaluate(env, policy, episodes=100):
    wins = 0
    total_steps = 0
    for _ in range(episodes):
        obs = env.reset()
        while not obs.done:
            action = policy.select_action(obs)
            obs = env.step(action)
        if obs.reward and obs.reward > 0:
            wins += 1
        total_steps += env.state.step_count
    return wins / episodes, total_steps / episodes


# --- Main execution block for inference.py ---
if __name__ == "__main__":
    print("Running inference.py...")
    env = WordGameEnvironment()

    for policy in [RandomLetterPolicy(), FrequencyPolicy()]:
        win_rate, avg_steps = evaluate(env, policy)
        print(f"{policy.name:15s} — Win rate: {win_rate*100:.1f}%, Avg steps: {avg_steps:.1f}")


Overwriting inference.py


In [ ]:
import os

inference_content = '''
import random
import uuid
import string
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

# --- Word Game Type Definitions (from 5PJ2MVnlV7xP) ---
@dataclass
class WordGameAction:
    """Player guesses a single letter."""
    guess: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameObservation:
    """What the player sees after each guess."""
    done: bool
    reward: Optional[float]
    masked_word: str            # e.g., "p_th_n"
    guessed_letters: List[str]  # All letters tried
    attempts_remaining: int
    message: str                # Feedback text
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameState:
    """Episode metadata."""
    episode_id: Optional[str] = None
    step_count: int = 0
    target_word: str = ""
    max_attempts: int = 6

# --- Word Game Environment (from 1Zrghkx_V7xR) ---
WORDS = [
    "python", "neural", "tensor", "matrix", "vector",
    "kernel", "lambda", "signal", "binary", "cipher",
    "model", "layer", "epoch", "batch", "token",
]

class WordGameEnvironment:
    """A letter-guessing game environment following the OpenEnv pattern."""

    def __init__(self):
        self._state = WordGameState()
        self._target = ""
        self._guessed = set()
        self._remaining = 6

    def reset(self) -> WordGameObservation:
        """Start a new episode with a random word."""
        self._target = random.choice(WORDS)
        self._guessed = set()
        self._remaining = 6
        self._state = WordGameState(
            episode_id=str(uuid.uuid4()),
            step_count=0,
            target_word=self._target,
            max_attempts=6,
        )
        return WordGameObservation(
            done=False,
            reward=None,
            masked_word=self._mask(),
            guessed_letters=[],
            attempts_remaining=self._remaining,
            message=f"Guess letters in a {len(self._target)}-letter word!",
        )

    def step(self, action: WordGameAction) -> WordGameObservation:
        """Process a letter guess."""
        letter = action.guess.lower().strip()
        self._state.step_count += 1

        # Already guessed?
        if letter in self._guessed:
            return WordGameObservation(
                done=False,
                reward=0.0,
                masked_word=self._mask(),
                guessed_letters=sorted(self._guessed),
                attempts_remaining=self._remaining,
                message=f"Already guessed '{letter}'. Try another.",
            )

        self._guessed.add(letter)

        if letter in self._target:
            message = f"'{letter}' is in the word!"
        else:
            self._remaining -= 1
            message=f"'{letter}' is not in the word."

        # Check win/lose
        masked = self._mask()
        won = "_" not in masked
        lost = self._remaining <= 0
        done = won or lost

        if won:
            reward = 1.0
            message = f"You got it! The word was '{self._target}'."
        elif lost:
            reward = 0.0
            message = f"Out of attempts. The word was '{self._target}'."
        else:
            reward = 0.0

        return WordGameObservation(
            done=done,
            reward=reward,
            masked_word=masked,
            guessed_letters=sorted(self._guessed),
            attempts_remaining=self._remaining,
            message=message,
        )

    @property
    def state(self) -> WordGameState:
        return self._state

    def _mask(self) -> str:
        """Show guessed letters, hide the rest."""
        return "".join(c if c in self._guessed else "_" for c in self._target)

# --- Policy Definitions and Evaluation Function (from 52aizQEBV7xU) ---
class RandomLetterPolicy:
    """Guess random unused letters."""
    name = "Random"

    def select_action(self, obs: WordGameObservation) -> WordGameAction:
        available = [c for c in string.ascii_lowercase if c not in obs.guessed_letters]
        return WordGameAction(guess=random.choice(available))


class FrequencyPolicy:
    """Guess by English letter frequency."""
    name = "Frequency"
    FREQ_ORDER = "etaoinshrdlcumwfgypbvkjxqz"

    def select_action(self, obs: WordGameObservation) -> WordGameAction:
        for letter in self.FREQ_ORDER:
            if letter not in obs.guessed_letters:
                return WordGameAction(guess=letter)
        return WordGameAction(guess="a")  # fallback


def evaluate(env, policy, episodes=100):
    wins = 0
    total_steps = 0
    for _ in range(episodes):
        obs = env.reset()
        while not obs.done:
            action = policy.select_action(obs)
            obs = env.step(action)
        if obs.reward and obs.reward > 0:
            wins += 1
        total_steps += env.state.step_count
    return wins / episodes, total_steps / episodes


# --- Main execution block for inference.py ---
if __name__ == "__main__":
    print("Running inference.py...")
    env = WordGameEnvironment()

    for policy in [RandomLetterPolicy(), FrequencyPolicy()]:
        win_rate, avg_steps = evaluate(env, policy)
        print(f"{policy.name:15s} — Win rate: {win_rate*100:.1f}%, Avg steps: {avg_steps:.1f}")
'''

# Write the inference.py file
with open("inference.py", "w") as f:
    f.write(inference_content)

# Execute the inference.py file
!python inference.py


Running inference.py...
Random          — Win rate: 2.0%, Avg steps: 7.6
Frequency       — Win rate: 4.0%, Avg steps: 9.2


In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

# These would normally go in models.py

@dataclass
class WordGameAction:
    """Player guesses a single letter."""
    guess: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameObservation:
    """What the player sees after each guess."""
    done: bool
    reward: Optional[float]
    masked_word: str            # e.g., "p_th_n"
    guessed_letters: List[str]  # All letters tried
    attempts_remaining: int
    message: str                # Feedback text
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameState:
    """Episode metadata."""
    episode_id: Optional[str] = None
    step_count: int = 0
    target_word: str = ""
    max_attempts: int = 6

print("Types defined: WordGameAction, WordGameObservation, WordGameState")

Types defined: WordGameAction, WordGameObservation, WordGameState


## 2. Implement the Environment

The environment implements three methods: `reset()`, `step()`, and `state`. This is where the game logic lives.

In [ ]:
import random
import uuid

WORDS = [
    "python", "neural", "tensor", "matrix", "vector",
    "kernel", "lambda", "signal", "binary", "cipher",
    "model", "layer", "epoch", "batch", "token",
]

class WordGameEnvironment:
    """A letter-guessing game environment following the OpenEnv pattern."""

    def __init__(self):
        self._state = WordGameState()
        self._target = ""
        self._guessed = set()
        self._remaining = 6

    def reset(self) -> WordGameObservation:
        """Start a new episode with a random word."""
        self._target = random.choice(WORDS)
        self._guessed = set()
        self._remaining = 6
        self._state = WordGameState(
            episode_id=str(uuid.uuid4()),
            step_count=0,
            target_word=self._target,
            max_attempts=6,
        )
        return WordGameObservation(
            done=False,
            reward=None,
            masked_word=self._mask(),
            guessed_letters=[],
            attempts_remaining=self._remaining,
            message=f"Guess letters in a {len(self._target)}-letter word!",
        )

    def step(self, action: WordGameAction) -> WordGameObservation:
        """Process a letter guess."""
        letter = action.guess.lower().strip()
        self._state.step_count += 1

        # Already guessed?
        if letter in self._guessed:
            return WordGameObservation(
                done=False,
                reward=0.0,
                masked_word=self._mask(),
                guessed_letters=sorted(self._guessed),
                attempts_remaining=self._remaining,
                message=f"Already guessed '{letter}'. Try another.",
            )

        self._guessed.add(letter)

        if letter in self._target:
            message = f"'{letter}' is in the word!"
        else:
            self._remaining -= 1
            message = f"'{letter}' is not in the word."

        # Check win/lose
        masked = self._mask()
        won = "_" not in masked
        lost = self._remaining <= 0
        done = won or lost

        if won:
            reward = 1.0
            message = f"You got it! The word was '{self._target}'."
        elif lost:
            reward = 0.0
            message = f"Out of attempts. The word was '{self._target}'."
        else:
            reward = 0.0

        return WordGameObservation(
            done=done,
            reward=reward,
            masked_word=masked,
            guessed_letters=sorted(self._guessed),
            attempts_remaining=self._remaining,
            message=message,
        )

    @property
    def state(self) -> WordGameState:
        return self._state

    def _mask(self) -> str:
        """Show guessed letters, hide the rest."""
        return "".join(c if c in self._guessed else "_" for c in self._target)

print("WordGameEnvironment defined.")

WordGameEnvironment defined.


In [ ]:
env = WordGameEnvironment()
obs = env.reset()
print(f"Word: {obs.masked_word} ({len(obs.masked_word)} letters)")
print(f"Message: {obs.message}")
print(f"Attempts: {obs.attempts_remaining}")
print()

# Play with common letters
for letter in ["e", "a", "t", "n", "o", "r", "s", "i", "l"]:
    if obs.done:
        break
    obs = env.step(WordGameAction(guess=letter))
    print(f"  Guess '{letter}': {obs.masked_word}  ({obs.message})")

print(f"\nFinal: reward={obs.reward}, done={obs.done}")
print(f"State: episode={env.state.episode_id[:8]}..., steps={env.state.step_count}")

Word: ______ (6 letters)
Message: Guess letters in a 6-letter word!
Attempts: 6

  Guess 'e': _e__e_  ('e' is in the word!)
  Guess 'a': _e__e_  ('a' is not in the word.)
  Guess 't': _e__e_  ('t' is not in the word.)
  Guess 'n': _e_ne_  ('n' is in the word!)
  Guess 'o': _e_ne_  ('o' is not in the word.)
  Guess 'r': _erne_  ('r' is in the word!)
  Guess 's': _erne_  ('s' is not in the word.)
  Guess 'i': _erne_  ('i' is not in the word.)
  Guess 'l': _ernel  ('l' is in the word!)

Final: reward=0.0, done=False
State: episode=82f33f7a..., steps=9


## 4. Write Policies

Let's write two policies and compare them.

In [ ]:
import string

class RandomLetterPolicy:
    """Guess random unused letters."""
    name = "Random"

    def select_action(self, obs: WordGameObservation) -> WordGameAction:
        available = [c for c in string.ascii_lowercase if c not in obs.guessed_letters]
        return WordGameAction(guess=random.choice(available))


class FrequencyPolicy:
    """Guess by English letter frequency."""
    name = "Frequency"
    FREQ_ORDER = "etaoinshrdlcumwfgypbvkjxqz"

    def select_action(self, obs: WordGameObservation) -> WordGameAction:
        for letter in self.FREQ_ORDER:
            if letter not in obs.guessed_letters:
                return WordGameAction(guess=letter)
        return WordGameAction(guess="a")  # fallback


def evaluate(env, policy, episodes=100):
    wins = 0
    total_steps = 0
    for _ in range(episodes):
        obs = env.reset()
        while not obs.done:
            action = policy.select_action(obs)
            obs = env.step(action)
        if obs.reward and obs.reward > 0:
            wins += 1
        total_steps += env.state.step_count
    return wins / episodes, total_steps / episodes


env = WordGameEnvironment()

for policy in [RandomLetterPolicy(), FrequencyPolicy()]:
    win_rate, avg_steps = evaluate(env, policy)
    print(f"{policy.name:15s} — Win rate: {win_rate*100:.1f}%, Avg steps: {avg_steps:.1f}")

Random          — Win rate: 0.0%, Avg steps: 7.4
Frequency       — Win rate: 8.0%, Avg steps: 9.2


Wire Up FastAPI

In [ ]:
from openenv.core.env_server import create_fastapi_app

app = create_fastapi_app(WordGameEnvironment, WordGameAction, WordGameObservation)

In [ ]:
# Write the environment files to disk
import os

os.makedirs("word_game/server", exist_ok=True)

# models.py
models_code = '''
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

@dataclass
class WordGameAction:
    guess: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameObservation:
    done: bool
    reward: Optional[float]
    masked_word: str
    guessed_letters: List[str]
    attempts_remaining: int
    message: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class WordGameState:
    episode_id: Optional[str] = None
    step_count: int = 0
    target_word: str = ""
    max_attempts: int = 6
'''

with open("word_game/models.py", "w") as f:
    f.write(models_code)

print("Created word_game/models.py")
print("Created word_game/server/")
print("\nTo deploy:")
print("  1. Add server/environment.py (the WordGameEnvironment class above)")
print("  2. Add server/app.py with create_fastapi_app(WordGameEnvironment)")
print("  3. Add client.py with HTTPEnvClient subclass")
print("  4. Run: openenv push --repo-id username/word-game")

Created word_game/models.py
Created word_game/server/

To deploy:
  1. Add server/environment.py (the WordGameEnvironment class above)
  2. Add server/app.py with create_fastapi_app(WordGameEnvironment)
  3. Add client.py with HTTPEnvClient subclass
  4. Run: openenv push --repo-id username/word-game


In [ ]:
# Install dependencies
!uv pip install -q openenv-core==0.2.2

In [ ]:
# ✅ Define Action
class WordGameAction:
    def __init__(self, guess):
        self.guess = guess


# ✅ Define Observation
class WordGameObservation:
    def __init__(self, feedback, attempts_left):
        self.feedback = feedback
        self.attempts_left = attempts_left


# ✅ Define State
class WordGameState:
    def __init__(self, secret_word, attempts_left):
        self.secret_word = secret_word
        self.attempts_left = attempts_left


# ✅ Local Word Game Environment (NO SERVER)
class WordGameEnv:
    def __init__(self):
        self.secret_word = "apple"
        self.max_attempts = 5
        self.reset()

    def reset(self):
        self.state = WordGameState(self.secret_word, self.max_attempts)
        return WordGameObservation("Game started", self.state.attempts_left)

    def step(self, action):
        guess = action.guess.lower()
        self.state.attempts_left -= 1

        if guess == self.secret_word:
            return {
                "observation": WordGameObservation("Correct!", self.state.attempts_left),
                "reward": 1,
                "done": True
            }

        # simple feedback
        feedback = "".join([
            g if g == s else "_"
            for g, s in zip(guess, self.secret_word)
        ])

        done = self.state.attempts_left == 0

        return {
            "observation": WordGameObservation(feedback, self.state.attempts_left),
            "reward": 0,
            "done": done
        }


# ✅ Run the environment (Colab safe)
env = WordGameEnv()

obs = env.reset()
print("Start:", obs.feedback)

guesses = ["grape", "apron", "apple"]

for g in guesses:
    result = env.step(WordGameAction(g))
    print(f"\nGuess: {g}")
    print("Feedback:", result["observation"].feedback)
    print("Attempts left:", result["observation"].attempts_left)

    if result["done"]:
        print("Game Over!")
        break

Start: Game started

Guess: grape
Feedback: ____e
Attempts left: 4

Guess: apron
Feedback: ap___
Attempts left: 3

Guess: apple
Feedback: Correct!
Attempts left: 2
Game Over!


In [ ]:
# ✅ Action class
class WordGameAction:
    def __init__(self, guess):
        self.guess = guess


# ✅ Observation class
class WordGameObservation:
    def __init__(self, masked_word, attempts_left):
        self.masked_word = masked_word
        self.attempts_left = attempts_left


# ✅ Local WordGame Environment
class WordGameEnv:
    def __init__(self):
        self.secret_word = "apple"
        self.max_attempts = 6
        self.reset()

    def reset(self):
        self.guessed_letters = set()
        self.attempts_left = self.max_attempts
        return self._get_observation()

    def step(self, action):
        letter = action.guess.lower()

        if letter not in self.guessed_letters:
            self.guessed_letters.add(letter)

            if letter not in self.secret_word:
                self.attempts_left -= 1

        done = self.attempts_left == 0 or self._is_word_guessed()

        return type("StepResult", (), {
            "observation": self._get_observation(),
            "reward": 1 if self._is_word_guessed() else 0,
            "done": done
        })

    def _get_observation(self):
        masked = "".join([
            c if c in self.guessed_letters else "_"
            for c in self.secret_word
        ])
        return WordGameObservation(masked, self.attempts_left)

    def _is_word_guessed(self):
        return all(c in self.guessed_letters for c in self.secret_word)


# ✅ Run (same style as your original)
env = WordGameEnv()

result = env.reset()
print("Start:", result.masked_word)

result = env.step(WordGameAction(guess="e"))
print("After guess 'e':", result.observation.masked_word)

Start: _____
After guess 'e': ____e


In [ ]:
!rm -rf word_game
!openenv init word_game
# cd word_game # 'cd' needs its own cell or context management
# Edit models.py, server/environment.py, client.py
# uv run server           # Test locally
# openenv push             # Deploy to HF Spaces

Creating OpenEnv environment 'word_game'...
✓ Created 11 files

Generating uv.lock...
✓ Generated uv.lock

Environment created successfully at: /content/word_game

Next steps:
  cd /content/word_game
  # Edit your environment implementation in server/word_game_environment.py
  # Edit your models in models.py
  # Install dependencies: uv sync

  # To integrate into OpenEnv repo:
  # 1. Copy this directory to <repo_root>/envs/word_game_env
  # 2. Build from repo root: docker build -t word_game_env:latest -f 
envs/word_game_env/server/Dockerfile .
  # 3. Run your image: docker run -p 8000:8000 word_game_env:latest


## Summary

You built a complete OpenEnv environment:

| File | What it does | Lines of code |
|------|-------------|---------------|
| `models.py` | Action, Observation, State types | ~30 |
| `server/environment.py` | Game logic (reset, step, state) | ~60 |
| `client.py` | HTTP client (3 parsing methods) | ~25 |
| `server/app.py` | FastAPI wiring | ~3 |

The pattern is always the same: **types → server logic → client → container**.

**Next:** [Module 5](../module-5/README.md) — Training a model to play games with GRPO.